<div style="background:#1E40AF; padding: 40px; border-radius: 10px; color: white; font-family: sans-serif;">
    <h1 style="margin:0;">Dashboard Analítico</h1>
    <p style="margin:5px 0 0; opacity:0.8;">Gestión de Empleados y Proyectos | Future Space</p>
</div>

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from data_engine import DataEngine
from IPython.display import display, HTML

C = {
    'primary':     '#1E40AF', 
    'text':        '#FFFFFF', 
    'muted':       '#94A3B8',
    'success':     '#10B981',
    'danger':      '#EF4444',
}
FONT = 'Inter, sans-serif'
FD   = 'Outfit, sans-serif'

_base = dict(template='plotly_white', plot_bgcolor='white', paper_bgcolor='white', margin=dict(l=80, r=50, t=140, b=80))

def _style(fig, title, sub='', h=400, hbar=False):
    t = f'<b style="font-family:{FD};font-size:18px;color:#0F172A;">{title}</b>'
    if sub: t += f'<br><span style="font-size:12px;color:#64748B;">{sub}</span>'
    fig.update_layout(**_base, height=h, title=dict(text=t, x=0.02, y=0.88))
    fig.update_xaxes(gridcolor=\'#F1F5F9\', zeroline=False)
    fig.update_yaxes(gridcolor=\'#F1F5F9\', zeroline=False)
    return fig

def section(icon, title, sub=''):
    display(HTML(f'''
        <div style="margin:70px 0 30px; font-family:{FONT};">
            <div style="display:flex; align-items:center; gap:15px; margin-bottom:10px;">
                <div style="font-size:32px;">{icon}</div>
                <div>
                    <h2 style="margin:0; color:#FFFFFF; font-size:24px; font-weight:800;">{title}</h2>
                    <p style="margin:2px 0 0; color:#94A3B8; font-size:14px;">{sub}</p>
                </div>
            </div>
            <div style="height:2px; background:#1E40AF; width:100px; opacity:0.6;"></div>
        </div>
    '''))

def note(text):
    display(HTML(f'<div style="background:#F8FAFC; border-left:4px solid #1E40AF; padding:15px; color:#334155; margin:20px 0; border-radius:4px; font-size:14px;"><b>Observaciones:</b> {text}</div>'))

print('✅ Sistema cargado.')

SyntaxError: unexpected character after line continuation character (805629050.py, line 24)

In [ ]:
engine = DataEngine()
df_emp = engine.get_employees()
df_prj = engine.get_projects()
df_asg = engine.get_assignments()

df_emp['Antigüedad'] = df_emp['ANTIGUEDAD_ANOS'].round(1)
df_emp['Edad'] = df_emp['EDAD'].astype(int)
df_emp['AnioAlta'] = pd.to_datetime(df_emp['F_ALTA']).dt.year
df_emp['AnioBaja'] = pd.to_datetime(df_emp['F_BAJA']).dt.year
activos = df_emp[df_emp['ESTADO'] == 'Activo'].copy()
prj_act = df_prj[df_prj['ESTADO'] == 'Activo'].copy()

print(f'Datos listos. Empleados: {len(activos)} | Proyectos: {len(prj_act)}')

In [ ]:
section('👥', 'Empleados | Antigüedad', 'Análisis de veteranía en la empresa')
fig = px.histogram(activos, x='Antigüedad', nbins=10, color_discrete_sequence=[C['primary']])
_style(fig, 'Años en la empresa', 'Distribución de la plantilla por tiempo trabajado')
fig.show()
note('La antigüedad media refleja una plantilla estable y consolidada.')

In [ ]:
section('🎂', 'Empleados | Distribución de Edad', 'Reparto de la plantilla por rangos de edad')
fig = px.histogram(activos, x='Edad', nbins=12, color_discrete_sequence=[C['primary']])
_style(fig, 'Distribución por edades', 'Histograma de empleados en activo')
fig.show()
note('El perfil de edad muestra una alta madurez profesional en el equipo.')

In [ ]:
section('📈', 'Evolución de Plantilla', 'Histórico de altas y bajas por año')
altas = df_emp.groupby('AnioAlta').size().reset_index(name='Altas')
bajas = df_emp.dropna(subset=['AnioBaja']).groupby('AnioBaja').size().reset_index(name='Bajas')
evol = pd.merge(altas, bajas, left_on='AnioAlta', right_on='AnioBaja', how='outer').fillna(0)
evol['Año'] = evol['AnioAlta'].combine_first(evol['AnioBaja']).astype(int)
evol['Neto'] = evol['Altas'] - evol['Bajas']

fig = go.Figure()
fig.add_trace(go.Bar(x=evol['Año'], y=evol['Altas'], name='Altas', marker_color=C['success']))
fig.add_trace(go.Bar(x=evol['Año'], y=-evol['Bajas'], name='Bajas', marker_color=C['danger']))
_style(fig, 'Evolución anual', 'Contrataciones vs Bajas registradas')
fig.show()

best_y = int(evol.loc[evol['Neto'].idxmax()]['Año'])
worst_y = int(evol.loc[evol['Neto'].idxmin()]['Año'])
display(HTML(f'<div style="padding:15px; background:#F1F5F9; border-radius:8px;">Año de mayor crecimiento: <b>{best_y}</b> | Año de mayor descenso: <b>{worst_y}</b></div>'))

In [ ]:
section('📁', 'Proyectos', 'Estado y distribución del portfolio')
fig = px.bar(df_prj.groupby('Sede').size().reset_index(name='Total'), x='Total', y='Sede', orientation='h', color_discrete_sequence=[C['primary']])
_style(fig, 'Proyectos por Sede', 'Distribución geográfica del trabajo')
fig.show()
note(f'Actualmente contamos con {len(prj_act)} proyectos activos en total.')

In [ ]:
section('⚖️', 'Carga de Trabajo', 'Asignación de recursos por empleado')
emp_load = df_asg.groupby('ID_EMPLEADO').size().reset_index(name='Proyectos')
fig = px.bar(emp_load.groupby('Proyectos').size().reset_index(name='Empleados'), x='Proyectos', y='Empleados', color_discrete_sequence=[C['primary']])
_style(fig, 'Asignaciones por persona', 'Distribución de carga operativa')
fig.show()
note('La asignación de recursos se mantiene dentro de los límites operativos.')